In [14]:
from langchain.chat_models import init_chat_model
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition

In [15]:
from dotenv import load_dotenv
load_dotenv()

True

In [16]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()

In [17]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [18]:
@tool
def get_stock_price(symbol: str) -> float:
    '''Return the current price of a stock given the stock symbol (whenever a query asks for stock prices reference this function)
    :param symbol: stock symbol
    :return: current price of the stock
    '''
    return {
        "MSFT": 200.3,
        "AAPL": 100.4,
        "AMZN": 150.0,
        "RIL": 87.6
    }.get(symbol, 0.0)
tools = [get_stock_price]

llm = init_chat_model("google_genai:gemini-3.5-flash")
llm_with_tools = llm.bind_tools(tools)

In [19]:
def chatbot(state: State) -> State:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

builder = StateGraph(State)

builder.add_node(chatbot)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "chatbot")
builder.add_conditional_edges("chatbot", tools_condition)
builder.add_edge("tools", "chatbot")
builder.add_edge("chatbot", END)  # Optionally needed (can keep for comprehension)

graph = builder.compile(checkpointer=memory)

#Display graph
from IPython.display import display, Markdown

mermaid_text = graph.get_graph().draw_mermaid()
display(Markdown(f"```mermaid\n{mermaid_text}\n```"))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	chatbot(chatbot)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> chatbot;
	chatbot -.-> __end__;
	chatbot -.-> tools;
	tools --> chatbot;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [20]:
config1 = { "configurable": { "thread_id": 1 } }
msg = "I want to buy 20 AMZN stock using the current price. Then 15 MSFT. What will be the total cost?"

state = graph.invoke({"messages": [{"role": "user", "content": msg}]}, config=config1)
print(state["messages"][-1].content)

[{'type': 'text', 'text': 'The total cost to buy 20 shares of AMZN and 15 shares of MSFT at their current prices is **$6,004.50**.\n\nHere is the breakdown:\n* **AMZN:** 20 shares × $150.00 = $3,000.00\n* **MSFT:** 15 shares × $200.30 = $3,004.50\n* **Total:** $3,000.00 + $3,004.50 = $6,004.50', 'extras': {'signature': 'Er0CCroCAQw51sejn9y6oPwEfFP1EIFz07HPOLUcR8ma/dv3dnfH7I9uE87XNnczKe6KDVvIODlJl8190uCN08xlZ2yLKG8F5i+gDF3WUDJRyIpx9PWaqei3+GNHK9G58IpJupKbiECbEWTt8LnB3nLYlaRnCz4tIFSzKJpVLhZeOMdgrPJ5/c4oXaYTtXtwOusxmtXkyUOYEQQIfcqXFoX7WIeI9XUiXAW+7t2Om0BnsQ51I63A6KkwNj6KW1uPGffGxYRzklvwnFFtTiMO15oKeYZKrXcb6sCwzDy/KfYkb9qj2NE+lFiJ8k0o5+ORN1B3q9kxuDmbZeluuA7x2L8lCOJSKXSiidAb2FPjcAhMwKMzC//UxJEOf0rOUn0JhkFsB9zR/fj/55PhBedZ551KLFQq0xo8vGFVAzYKHdQ='}}]


In [21]:
config2 = { "configurable": { "thread_id": 2 } }
msg = "Tell me the current price of 5 AAPL stocks."

state = graph.invoke({"messages": [{"role": "user", "content": msg}]}, config=config2)
print(state["messages"][-1].content)

[{'type': 'text', 'text': 'The current price of one AAPL stock is $100.40. Therefore, the price for 5 AAPL stocks is $502.00.', 'extras': {'signature': 'Ev8BCvwBAQw51sfSFJsuxH1vAAR5JVZy93CaAxuf+ISS44EBTfwfoexmymUHTbKc7V6yOTGFtDRGqXgTjdK9mc8j7x08b1qp0vUvu/WWycch4GIThvpBYWekCdOwPnzK6+5vRtEybrBcBHoM7Td3U9FBlJq4lz1UyAQKMcpcWnrRSAL+wCg/Oi6+C4uD9LcBGEZcnuHpIdXpby+C0v7ggHfqDlLT9QIQbs0PMpc21uW1jRS0o9i+awdLlUjAHeV0XDPCJzIFX4lW+czzKyYmQI27EPs/S5gOK24GN4L5YLv8e6ykrfdwRaIBOEXhI7Wdci8HRN4tMq79AIRwkWhGonw1'}}]


In [22]:
msg = "Using the current price, tell me the total price of 10 RIL stocks and add it to previous total"

state = graph.invoke({"messages": [{"role": "user", "content": msg}]}, config=config1)
print(state["messages"][-1].content)

[{'type': 'text', 'text': 'The total cost for 10 RIL stocks at the current price of $87.60 is **$876.00**.\n\nAdding this to your previous total of $6,004.50 brings the new total to **$6,880.50**.', 'extras': {'signature': 'EoYCCoMCAQw51seDqyGEZMMDVeuM0ERUW7pJpabex+8OVDR/pyYdMYqabROBamFEDH/SuD3yyscet/D339IiCxskPES8jZx/32W5brrKpBgynAbNJsGqI3RMBRdM34ZColYb2BF8mPAbbN+hifvThGXPVD0N43EqlCWrlDSl71zgv1Qa2CpF3nDaKjq9wNVaozqn6XYe1WWXto2ZD6rsrJFFIs6fspkL92BtEdI/f+8TH+c9aOOr7k4XJupyP/XlT6vadnjC+y2a3iLa3m26zitL8OIM4sM7F1pDDUj5j9fQZKBcI++XX1B0NVDqSy12S8LlkTmJRdMDWrhd6Lyz0YbtHHqmsuOhMw=='}}]


In [24]:
msg = "Tell me the current price of 5 MSFT stocks and add it to previous total"

state = graph.invoke({"messages": [{"role": "user", "content": msg}]}, config=config2)
print(state["messages"][-1].content)

[{'type': 'text', 'text': 'The current price of one MSFT stock is $200.30. \n\nHere is the calculation:\n* Price for 5 MSFT stocks: 5 * $200.30 = $1,001.50\n* Previous total (5 AAPL stocks): $502.00\n\nAdding these together, the new total is **$1,503.50**.', 'extras': {'signature': 'EosECogEAQw51sedecy5XQLkwRFUAy+VI52oenVei1rwsvO4uhWJJHb5RqiPLcQBEgw0e5lVsTZxylNBkt/GZEZoSVcTN57r/yN1XpV5L9horqC/GFdYFpjr02Pgs2wV2ZNWxLv0JIDWw2ki2BWb58Lm662x3/gRxqFiQALqzrKkiESPF4wRCVUC8gpc92vCAWujhrPkK0X97hHQknm6OJK63c1JDTwcP98O9qSp4OmN/QAUK7DFXJ51Pvn+TGlvSXgjHCmEZMylUon/KWZgOTb90JEBbPQyGJF7k44JIwFB0REYaNB2euIB+6VPs7758ax5b/vVPau6rqBGRHnzHBcpfoHgelfXIkYUc9haM+N3U56+o9gz+pD0WfGcbXaP0hc4AznLQ8Kva3UoeEAs/4Q20x4V12gsNIjo5wXJdutEfW5nMUu1FrT2wpNF5CTWhvjMv62ZV/4jz498DsZJyZqLVsy5Jmpl3z8I5IrSw/uaWXwc25cjAjgNOhCR2JUMRjh4vD7+x9yaHFP1n3BTLnPeZdpdCyQ+mNj72R4H5a79ql3eVauqYYgn36kVQDrl+XZj5SBgxJm9HfZ/Hc+0HDIZfA62pBmsYWFLGC6UI86cLlWyRaRuOWDoFxUtN0BBFgaLJIYHe66eL35mqSlo+UjIWtw9whocWjOmsUl41LKDJjecCUZj4Rdbji8ixQ=='}}]
